In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
dbutils.widgets.text(
    "catalogo",
    "catalog_au"
)

dbutils.widgets.text(
    "esquema_source",
    "silver"
)

dbutils.widgets.text(
    "esquema_sink",
    "golden"
)

In [0]:
catalogo = dbutils.widgets.get(
    "catalogo"
)

esquema_source = dbutils.widgets.get(
    "esquema_source"
)

esquema_sink = dbutils.widgets.get(
    "esquema_sink"
)

In [0]:
df_mobility_events = spark.table(
    f"{catalogo}.{esquema_source}.mobility_events"
)

In [0]:
# ============================================================
# PREPARAR FUENTE PARA GOLDEN
# ============================================================

df_mobility_valid = (
    df_mobility_events

    .filter(
        col("dq_valid_event") == 1
    )
)

# ============================================================
# VALIDAR FUENTE GOLDEN
# ============================================================

df_mobility_valid.groupBy(
    "transport_type"
).agg(
    count("*")
        .alias("total_events")
).orderBy(
    "transport_type"
).show()

+--------------+------------+
|transport_type|total_events|
+--------------+------------+
|          BIKE|     5675914|
|          TAXI|     1454641|
+--------------+------------+



In [0]:
# ============================================================
# CONSTRUIR GOLDEN MOBILITY HOURLY
# ============================================================

df_mobility_hourly = (
    df_mobility_valid

    .groupBy(
        col("event_date"),
        col("event_hour"),
        col("event_day_of_week"),
        col("event_month"),
        col("is_weekend"),

        col("start_location_id"),
        col("start_zone"),
        col("start_borough"),
        col("start_borough_category"),

        col("transport_type")
    )

    .agg(
        count("*")
            .alias("total_trips"),

        avg("duration_minutes")
            .alias("avg_duration_minutes"),

        min("duration_minutes")
            .alias("min_duration_minutes"),

        max("duration_minutes")
            .alias("max_duration_minutes")
    )

    .withColumn(
        "_load_timestamp",
        current_timestamp()
    )

    .orderBy(
        col("event_date"),
        col("event_hour"),
        col("start_location_id"),
        col("transport_type")
    )
)

In [0]:
# ============================================================
# VALIDAR MOBILITY HOURLY
# ============================================================

print(
    f"Registros Mobility Hourly: "
    f"{df_mobility_hourly.count():,}"
)

print(
    f"Columnas Mobility Hourly : "
    f"{len(df_mobility_hourly.columns)}"
)


df_mobility_hourly.select(
    count("*")
        .alias("total_filas"),

    sum("total_trips")
        .alias("total_trips"),

    countDistinct("event_date")
        .alias("dias"),

    countDistinct("event_hour")
        .alias("horas"),

    countDistinct("start_location_id")
        .alias("zonas"),

    countDistinct("transport_type")
        .alias("transport_types"),

    sum(
        when(
            col("start_location_id").isNull(),
            lit(1)
        ).otherwise(lit(0))
    ).alias("location_id_null")
).show(
    truncate=False
)

Registros Mobility Hourly: 490,915
Columnas Mobility Hourly : 15
+-----------+-----------+----+-----+-----+---------------+----------------+
|total_filas|total_trips|dias|horas|zonas|transport_types|location_id_null|
+-----------+-----------+----+-----+-----+---------------+----------------+
|490915     |7130555    |182 |24   |249  |2              |0               |
+-----------+-----------+----+-----+-----+---------------+----------------+



In [0]:
# ============================================================
# VALIDAR CONTRATO MOBILITY HOURLY
# ============================================================

tabla_mobility_hourly_golden = (
    f"{catalogo}.{esquema_sink}.mobility_hourly"
)


columnas_df = (
    df_mobility_hourly
    .columns
)

columnas_tabla = (
    spark.table(
        tabla_mobility_hourly_golden
    )
    .columns
)


print(
    f"Columnas DataFrame : {len(columnas_df)}"
)

print(
    f"Columnas tabla     : {len(columnas_tabla)}"
)


if columnas_df != columnas_tabla:
    raise Exception(
        "El orden o nombre de las columnas de "
        "Mobility Hourly no coincide con Golden."
    )


schema_df = [
    (
        field.name,
        field.dataType.simpleString()
    )
    for field
    in df_mobility_hourly.schema.fields
]


schema_tabla = [
    (
        field.name,
        field.dataType.simpleString()
    )
    for field
    in spark.table(
        tabla_mobility_hourly_golden
    ).schema.fields
]


if schema_df != schema_tabla:
    print("=== SCHEMA DATAFRAME ===")
    print(schema_df)

    print("=== SCHEMA TABLA ===")
    print(schema_tabla)

    raise Exception(
        "Los tipos de datos de Mobility Hourly "
        "no coinciden con Golden."
    )


print(
    "Contrato Mobility Hourly correcto."
)

Columnas DataFrame : 15
Columnas tabla     : 15
Contrato Mobility Hourly correcto.


In [0]:
# ============================================================
# ESCRIBIR Y RECONCILIAR MOBILITY HOURLY
# ============================================================

df_mobility_hourly.write\
    .mode("overwrite")\
    .insertInto(
        tabla_mobility_hourly_golden
    )


df_mobility_hourly_check = (
    spark.table(
        tabla_mobility_hourly_golden
    )
)


mobility_hourly_stats = (
    df_mobility_hourly_check

    .agg(
        count("*")
            .alias("total_filas"),

        sum("total_trips")
            .alias("total_trips"),

        min("event_date")
            .alias("fecha_min"),

        max("event_date")
            .alias("fecha_max")
    )
)


print(
    "=== RECONCILIACIÓN GOLDEN MOBILITY HOURLY ==="
)

mobility_hourly_stats.show(
    truncate=False
)

=== RECONCILIACIÓN GOLDEN MOBILITY HOURLY ===
+-----------+-----------+----------+----------+
|total_filas|total_trips|fecha_min |fecha_max |
+-----------+-----------+----------+----------+
|490915     |7130555    |2016-01-01|2016-06-30|
+-----------+-----------+----------+----------+



In [0]:
# ============================================================
# CONSTRUIR GOLDEN MOBILITY WEATHER
# ============================================================

df_mobility_weather = (
    df_mobility_valid

    .groupBy(
        col("event_date"),
        col("event_hour"),
        col("transport_type"),
        col("weather_condition"),
        col("has_precipitation"),
        col("has_rain")
    )

    .agg(
        count("*")
            .alias("total_trips"),

        avg("duration_minutes")
            .alias("avg_duration_minutes"),

        avg("temperature_2m_c")
            .alias("temperature_2m_c"),

        avg("precipitation_mm")
            .alias("precipitation_mm"),

        avg("rain_mm")
            .alias("rain_mm"),

        avg("cloudcover_pct")
            .alias("cloudcover_pct"),

        avg("windspeed_10m_kmh")
            .alias("windspeed_10m_kmh"),

        avg("winddirection_10m_deg")
            .alias("winddirection_10m_deg")
    )

    .withColumn(
        "_load_timestamp",
        current_timestamp()
    )

    .orderBy(
        col("event_date"),
        col("event_hour"),
        col("transport_type")
    )
)

In [0]:
# ============================================================
# VALIDAR MOBILITY WEATHER
# ============================================================

print(
    f"Registros Mobility Weather: "
    f"{df_mobility_weather.count():,}"
)

print(
    f"Columnas Mobility Weather : "
    f"{len(df_mobility_weather.columns)}"
)


df_mobility_weather.select(
    count("*")
        .alias("total_filas"),

    sum("total_trips")
        .alias("total_trips"),

    countDistinct("event_date")
        .alias("dias"),

    countDistinct("event_hour")
        .alias("horas"),

    countDistinct("transport_type")
        .alias("transport_types"),

    countDistinct("weather_condition")
        .alias("weather_conditions"),

    sum(
        when(
            col("temperature_2m_c").isNull(),
            lit(1)
        ).otherwise(lit(0))
    ).alias("temperature_null")
).show(
    truncate=False
)

Registros Mobility Weather: 8,626
Columnas Mobility Weather : 15
+-----------+-----------+----+-----+---------------+------------------+----------------+
|total_filas|total_trips|dias|horas|transport_types|weather_conditions|temperature_null|
+-----------+-----------+----+-----+---------------+------------------+----------------+
|8626       |7130555    |182 |24   |2              |3                 |0               |
+-----------+-----------+----+-----+---------------+------------------+----------------+



In [0]:
# ============================================================
# VALIDAR CONTRATO MOBILITY WEATHER
# ============================================================

tabla_mobility_weather_golden = (
    f"{catalogo}.{esquema_sink}.mobility_weather"
)


columnas_df = (
    df_mobility_weather
    .columns
)

columnas_tabla = (
    spark.table(
        tabla_mobility_weather_golden
    )
    .columns
)


print(
    f"Columnas DataFrame : {len(columnas_df)}"
)

print(
    f"Columnas tabla     : {len(columnas_tabla)}"
)


if columnas_df != columnas_tabla:
    raise Exception(
        "El orden o nombre de las columnas de "
        "Mobility Weather no coincide con Golden."
    )


schema_df = [
    (
        field.name,
        field.dataType.simpleString()
    )
    for field
    in df_mobility_weather.schema.fields
]


schema_tabla = [
    (
        field.name,
        field.dataType.simpleString()
    )
    for field
    in spark.table(
        tabla_mobility_weather_golden
    ).schema.fields
]


if schema_df != schema_tabla:
    print("=== SCHEMA DATAFRAME ===")
    print(schema_df)

    print("=== SCHEMA TABLA ===")
    print(schema_tabla)

    raise Exception(
        "Los tipos de datos de Mobility Weather "
        "no coinciden con Golden."
    )


print(
    "Contrato Mobility Weather correcto."
)

Columnas DataFrame : 15
Columnas tabla     : 15
Contrato Mobility Weather correcto.


In [0]:
# ============================================================
# ESCRIBIR Y RECONCILIAR MOBILITY WEATHER
# ============================================================

df_mobility_weather.write\
    .mode("overwrite")\
    .insertInto(
        tabla_mobility_weather_golden
    )


df_mobility_weather_check = (
    spark.table(
        tabla_mobility_weather_golden
    )
)


mobility_weather_stats = (
    df_mobility_weather_check

    .agg(
        count("*")
            .alias("total_filas"),

        sum("total_trips")
            .alias("total_trips"),

        min("event_date")
            .alias("fecha_min"),

        max("event_date")
            .alias("fecha_max"),

        sum(
            when(
                col("temperature_2m_c").isNull(),
                lit(1)
            ).otherwise(lit(0))
        ).alias("weather_null")
    )
)


print(
    "=== RECONCILIACIÓN GOLDEN MOBILITY WEATHER ==="
)

mobility_weather_stats.show(
    truncate=False
)

=== RECONCILIACIÓN GOLDEN MOBILITY WEATHER ===
+-----------+-----------+----------+----------+------------+
|total_filas|total_trips|fecha_min |fecha_max |weather_null|
+-----------+-----------+----------+----------+------------+
|8626       |7130555    |2016-01-01|2016-06-30|0           |
+-----------+-----------+----------+----------+------------+



In [0]:
# ============================================================
# MÉTRICAS BASE GOLDEN MOBILITY ZONE
# ============================================================

df_mobility_zone_base = (
    df_mobility_valid

    .groupBy(
        col("start_location_id"),
        col("start_zone"),
        col("start_borough"),
        col("start_borough_category"),
        col("transport_type")
    )

    .agg(
        count("*")
            .alias("total_trips"),

        countDistinct("event_date")
            .alias("active_days"),

        avg("duration_minutes")
            .alias("avg_duration_minutes"),

        sum(
            when(
                col("is_weekend") == 1,
                lit(1)
            ).otherwise(
                lit(0)
            )
        ).alias("weekend_trips"),

        sum(
            when(
                col("has_rain") == 1,
                lit(1)
            ).otherwise(
                lit(0)
            )
        ).alias("rainy_trips")
    )
)

In [0]:
# ============================================================
# CALCULAR PEAK HOUR POR ZONA Y TRANSPORTE
# ============================================================

df_zone_hourly = (
    df_mobility_valid

    .groupBy(
        col("start_location_id"),
        col("transport_type"),
        col("event_hour")
    )

    .agg(
        count("*")
            .alias("hour_trips")
    )
)


window_peak_hour = (
    Window

    .partitionBy(
        "start_location_id",
        "transport_type"
    )

    .orderBy(
        col("hour_trips").desc(),
        col("event_hour").asc()
    )
)


df_zone_peak_hour = (
    df_zone_hourly

    .withColumn(
        "rn",
        row_number().over(
            window_peak_hour
        )
    )

    .filter(
        col("rn") == 1
    )

    .select(
        col("start_location_id"),
        col("transport_type"),

        col("event_hour")
            .alias("peak_hour")
    )
)

In [0]:
# ============================================================
# CONSTRUIR GOLDEN MOBILITY ZONE
# ============================================================

df_mobility_zone = (
    df_mobility_zone_base.alias("b")

    .join(
        df_zone_peak_hour.alias("p"),

        (
            col("b.start_location_id")
            == col("p.start_location_id")
        )
        &
        (
            col("b.transport_type")
            == col("p.transport_type")
        ),

        "left"
    )

    .select(
        col("b.start_location_id"),
        col("b.start_zone"),
        col("b.start_borough"),
        col("b.start_borough_category"),
        col("b.transport_type"),

        col("b.total_trips"),
        col("b.active_days"),
        col("b.avg_duration_minutes"),
        col("b.weekend_trips"),
        col("b.rainy_trips"),

        col("p.peak_hour"),

        (
            col("b.total_trips")
            / col("b.active_days")
        ).cast("double")
         .alias("avg_trips_per_day")
    )

    .withColumn(
        "_load_timestamp",
        current_timestamp()
    )

    .orderBy(
        col("total_trips").desc()
    )
)

In [0]:
# ============================================================
# VALIDAR MOBILITY ZONE
# ============================================================

print(
    f"Registros Mobility Zone: "
    f"{df_mobility_zone.count():,}"
)

print(
    f"Columnas Mobility Zone : "
    f"{len(df_mobility_zone.columns)}"
)


df_mobility_zone.select(

    count("*")
        .alias("total_filas"),

    sum("total_trips")
        .alias("total_trips"),

    countDistinct("start_location_id")
        .alias("zonas"),

    countDistinct("transport_type")
        .alias("transport_types"),

    min("active_days")
        .alias("min_active_days"),

    max("active_days")
        .alias("max_active_days"),

    min("peak_hour")
        .alias("min_peak_hour"),

    max("peak_hour")
        .alias("max_peak_hour"),

    sum(
        when(
            col("peak_hour").isNull(),
            lit(1)
        ).otherwise(
            lit(0)
        )
    ).alias("peak_hour_null")

).show(
    truncate=False
)

Registros Mobility Zone: 321
Columnas Mobility Zone : 13
+-----------+-----------+-----+---------------+---------------+---------------+-------------+-------------+--------------+
|total_filas|total_trips|zonas|transport_types|min_active_days|max_active_days|min_peak_hour|max_peak_hour|peak_hour_null|
+-----------+-----------+-----+---------------+---------------+---------------+-------------+-------------+--------------+
|321        |7130555    |249  |2              |1              |182            |0            |23           |0             |
+-----------+-----------+-----+---------------+---------------+---------------+-------------+-------------+--------------+



In [0]:
# ============================================================
# VALIDAR CONTRATO MOBILITY ZONE
# ============================================================

tabla_mobility_zone_golden = (
    f"{catalogo}.{esquema_sink}.mobility_zone"
)


columnas_df = (
    df_mobility_zone
    .columns
)

columnas_tabla = (
    spark.table(
        tabla_mobility_zone_golden
    )
    .columns
)


print(
    f"Columnas DataFrame : {len(columnas_df)}"
)

print(
    f"Columnas tabla     : {len(columnas_tabla)}"
)


if columnas_df != columnas_tabla:
    raise Exception(
        "El orden o nombre de las columnas de "
        "Mobility Zone no coincide con Golden."
    )


schema_df = [
    (
        field.name,
        field.dataType.simpleString()
    )
    for field
    in df_mobility_zone.schema.fields
]


schema_tabla = [
    (
        field.name,
        field.dataType.simpleString()
    )
    for field
    in spark.table(
        tabla_mobility_zone_golden
    ).schema.fields
]


if schema_df != schema_tabla:

    print("=== SCHEMA DATAFRAME ===")
    print(schema_df)

    print("=== SCHEMA TABLA ===")
    print(schema_tabla)

    raise Exception(
        "Los tipos de datos de Mobility Zone "
        "no coinciden con Golden."
    )


print(
    "Contrato Mobility Zone correcto."
)

Columnas DataFrame : 13
Columnas tabla     : 13
Contrato Mobility Zone correcto.


In [0]:
# ============================================================
# ESCRIBIR Y RECONCILIAR MOBILITY ZONE
# ============================================================

df_mobility_zone.write\
    .mode("overwrite")\
    .insertInto(
        tabla_mobility_zone_golden
    )


df_mobility_zone_check = (
    spark.table(
        tabla_mobility_zone_golden
    )
)


print(
    "=== RECONCILIACIÓN GOLDEN MOBILITY ZONE ==="
)


df_mobility_zone_check.agg(

    count("*")
        .alias("total_filas"),

    sum("total_trips")
        .alias("total_trips"),

    countDistinct("start_location_id")
        .alias("zonas"),

    countDistinct("transport_type")
        .alias("transport_types"),

    sum(
        when(
            col("peak_hour").isNull(),
            lit(1)
        ).otherwise(
            lit(0)
        )
    ).alias("peak_hour_null")

).show(
    truncate=False
)

=== RECONCILIACIÓN GOLDEN MOBILITY ZONE ===
+-----------+-----------+-----+---------------+--------------+
|total_filas|total_trips|zonas|transport_types|peak_hour_null|
+-----------+-----------+-----+---------------+--------------+
|321        |7130555    |249  |2              |0             |
+-----------+-----------+-----+---------------+--------------+



In [0]:
# ============================================================
# CONSTRUIR GOLDEN TRANSPORT COMPARISON
# ============================================================

df_transport_comparison = (
    df_mobility_valid

    .groupBy(
        col("event_date"),
        col("event_hour"),
        col("event_day_of_week"),
        col("event_month"),
        col("is_weekend"),

        col("start_location_id"),
        col("start_zone"),
        col("start_borough"),
        col("start_borough_category"),

        col("temperature_2m_c"),
        col("precipitation_mm"),
        col("rain_mm"),
        col("weather_condition")
    )

    .agg(
        count("*")
            .alias("total_trips"),

        sum(
            when(
                col("transport_type") == "TAXI",
                lit(1)
            ).otherwise(lit(0))
        ).alias("taxi_trips"),

        sum(
            when(
                col("transport_type") == "BIKE",
                lit(1)
            ).otherwise(lit(0))
        ).alias("bike_trips"),

        avg(
            when(
                col("transport_type") == "TAXI",
                col("duration_minutes")
            )
        ).alias("avg_taxi_duration_minutes"),

        avg(
            when(
                col("transport_type") == "BIKE",
                col("duration_minutes")
            )
        ).alias("avg_bike_duration_minutes")
    )

    .withColumn(
        "taxi_share_pct",
        (
            col("taxi_trips")
            / col("total_trips")
        ) * lit(100.0)
    )

    .withColumn(
        "bike_share_pct",
        (
            col("bike_trips")
            / col("total_trips")
        ) * lit(100.0)
    )

    .withColumn(
        "_load_timestamp",
        current_timestamp()
    )

    .select(
        col("event_date"),
        col("event_hour"),
        col("event_day_of_week"),
        col("event_month"),
        col("is_weekend"),

        col("start_location_id"),
        col("start_zone"),
        col("start_borough"),
        col("start_borough_category"),

        col("total_trips"),
        col("taxi_trips"),
        col("bike_trips"),

        col("taxi_share_pct"),
        col("bike_share_pct"),

        col("avg_taxi_duration_minutes"),
        col("avg_bike_duration_minutes"),

        col("temperature_2m_c"),
        col("precipitation_mm"),
        col("rain_mm"),
        col("weather_condition"),

        col("_load_timestamp")
    )
)

In [0]:
# ============================================================
# VALIDAR TRANSPORT COMPARISON
# ============================================================

print(
    f"Registros Transport Comparison: "
    f"{df_transport_comparison.count():,}"
)

print(
    f"Columnas Transport Comparison : "
    f"{len(df_transport_comparison.columns)}"
)


df_transport_comparison.agg(

    count("*")
        .alias("total_filas"),

    sum("total_trips")
        .alias("total_trips"),

    sum("taxi_trips")
        .alias("taxi_trips"),

    sum("bike_trips")
        .alias("bike_trips"),

    countDistinct("event_date")
        .alias("dias"),

    countDistinct("event_hour")
        .alias("horas"),

    countDistinct("start_location_id")
        .alias("zonas"),

    sum(
        when(
            col("total_trips")
            != (
                col("taxi_trips")
                + col("bike_trips")
            ),
            lit(1)
        ).otherwise(lit(0))
    ).alias("invalid_trip_balance"),

    sum(
        when(
            abs(
                (
                    col("taxi_share_pct")
                    + col("bike_share_pct")
                ) - lit(100.0)
            ) > lit(0.000001),
            lit(1)
        ).otherwise(lit(0))
    ).alias("invalid_share_balance")

).show(
    truncate=False
)

Registros Transport Comparison: 316,412
Columnas Transport Comparison : 21
+-----------+-----------+----------+----------+----+-----+-----+--------------------+---------------------+
|total_filas|total_trips|taxi_trips|bike_trips|dias|horas|zonas|invalid_trip_balance|invalid_share_balance|
+-----------+-----------+----------+----------+----+-----+-----+--------------------+---------------------+
|316412     |7130555    |1454641   |5675914   |182 |24   |249  |0                   |0                    |
+-----------+-----------+----------+----------+----+-----+-----+--------------------+---------------------+



In [0]:
# ============================================================
# VALIDAR CONTRATO TRANSPORT COMPARISON
# ============================================================

tabla_transport_comparison_golden = (
    f"{catalogo}.{esquema_sink}.transport_comparison"
)


columnas_df = (
    df_transport_comparison
    .columns
)

columnas_tabla = (
    spark.table(
        tabla_transport_comparison_golden
    )
    .columns
)


print(
    f"Columnas DataFrame : {len(columnas_df)}"
)

print(
    f"Columnas tabla     : {len(columnas_tabla)}"
)


if columnas_df != columnas_tabla:
    raise Exception(
        "El orden o nombre de las columnas de "
        "Transport Comparison no coincide con Golden."
    )


schema_df = [
    (
        field.name,
        field.dataType.simpleString()
    )
    for field
    in df_transport_comparison.schema.fields
]


schema_tabla = [
    (
        field.name,
        field.dataType.simpleString()
    )
    for field
    in spark.table(
        tabla_transport_comparison_golden
    ).schema.fields
]


if schema_df != schema_tabla:

    print("=== SCHEMA DATAFRAME ===")
    print(schema_df)

    print("=== SCHEMA TABLA ===")
    print(schema_tabla)

    raise Exception(
        "Los tipos de datos de Transport Comparison "
        "no coinciden con Golden."
    )


print(
    "Contrato Transport Comparison correcto."
)

Columnas DataFrame : 21
Columnas tabla     : 21
Contrato Transport Comparison correcto.


In [0]:
# ============================================================
# ESCRIBIR Y RECONCILIAR TRANSPORT COMPARISON
# ============================================================

df_transport_comparison.write\
    .mode("overwrite")\
    .insertInto(
        tabla_transport_comparison_golden
    )


df_transport_comparison_check = (
    spark.table(
        tabla_transport_comparison_golden
    )
)


print(
    "=== RECONCILIACIÓN GOLDEN TRANSPORT COMPARISON ==="
)


df_transport_comparison_check.agg(

    count("*")
        .alias("total_filas"),

    sum("total_trips")
        .alias("total_trips"),

    sum("taxi_trips")
        .alias("taxi_trips"),

    sum("bike_trips")
        .alias("bike_trips"),

    min("event_date")
        .alias("fecha_min"),

    max("event_date")
        .alias("fecha_max"),

    countDistinct("start_location_id")
        .alias("zonas"),

    sum(
        when(
            col("total_trips")
            != (
                col("taxi_trips")
                + col("bike_trips")
            ),
            lit(1)
        ).otherwise(lit(0))
    ).alias("invalid_trip_balance")

).show(
    truncate=False
)

=== RECONCILIACIÓN GOLDEN TRANSPORT COMPARISON ===
+-----------+-----------+----------+----------+----------+----------+-----+--------------------+
|total_filas|total_trips|taxi_trips|bike_trips|fecha_min |fecha_max |zonas|invalid_trip_balance|
+-----------+-----------+----------+----------+----------+----------+-----+--------------------+
|316412     |7130555    |1454641   |5675914   |2016-01-01|2016-06-30|249  |0                   |
+-----------+-----------+----------+----------+----------+----------+-----+--------------------+



In [0]:
# ============================================================
# VALIDACIÓN FINAL GOLDEN
# ============================================================

tablas_golden = [
    "mobility_hourly",
    "mobility_weather",
    "mobility_zone",
    "transport_comparison"
]


print(
    "============================================"
)

print(
    "VALIDACIÓN FINAL CAPA GOLDEN"
)

print(
    "============================================"
)


for tabla in tablas_golden:

    nombre_tabla = (
        f"{catalogo}.{esquema_sink}.{tabla}"
    )

    registros = (
        spark.table(
            nombre_tabla
        )
        .count()
    )

    print(
        f"{nombre_tabla}: "
        f"{registros:,} registros"
    )


print()
print(
    "============================================"
)

print(
    "CARGA GOLDEN FINALIZADA CORRECTAMENTE"
)

print(
    "============================================"
)

VALIDACIÓN FINAL CAPA GOLDEN
catalog_au.golden.mobility_hourly: 490,915 registros
catalog_au.golden.mobility_weather: 8,626 registros
catalog_au.golden.mobility_zone: 321 registros
catalog_au.golden.transport_comparison: 316,412 registros

CARGA GOLDEN FINALIZADA CORRECTAMENTE
